In [ ]:
# ===============================================
# 1- Kütüphaneler ve Hazırlık
# ===============================================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
import os, zipfile, numpy as np
from PIL import Image
import matplotlib.pyplot as plt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Dataset Sınıfı (RGB olarak güncellendi)
class BrainTumorDataset(Dataset):
    def __init__(self, file_paths, labels, transform=None):
        self.file_paths, self.labels, self.transform = file_paths, labels, transform
    def __len__(self): return len(self.file_paths)
    def __getitem__(self, idx):
        # Isı haritası tutarlılığı için RGB okuma kritiktir
        image = Image.open(self.file_paths[idx]).convert("RGB")
        if self.transform: image = self.transform(image)
        return image, self.labels[idx]

# Veri Yollarını Çekme
def get_paths(root_dir):
    paths, labels = [], []
    classes = sorted(os.listdir(root_dir))
    for idx, cls in enumerate(classes):
        cls_dir = os.path.join(root_dir, cls)
        for img in os.listdir(cls_dir):
            paths.append(os.path.join(cls_dir, img))
            labels.append(idx)
    return paths, labels, classes

# Drive ve Zip İşlemleri
from google.colab import drive
drive.mount('/content/drive')
ZIP_PATH = "/content/drive/MyDrive/MRIBARIN.zip" # Kendi yolunu kontrol et

if not os.path.exists("/content/MRIBARIN"):
    with zipfile.ZipFile(ZIP_PATH, 'r') as z: z.extractall("/content/MRIBARIN")

train_paths, train_labels, class_names = get_paths("/content/MRIBARIN/Training")
val_paths, val_labels, _ = get_paths("/content/MRIBARIN/Testing")

# ===============================================
# 2- AKADEMİK ODAK İÇİN KRİTİK TRANSFORMLAR
# ===============================================
IMAGE_SIZE = 224


norm_mean = [0.485, 0.456, 0.406]
norm_std = [0.229, 0.224, 0.225]
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15)
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(norm_mean, norm_std)
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(norm_mean, norm_std) # TESTTE DE AYNI NORMALİZASYON OLMALI
])

# DataLoader Hazırlığı
train_ds = BrainTumorDataset(train_paths, train_labels, train_transform)
val_ds = BrainTumorDataset(val_paths, val_labels, val_transform)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)

# ===============================================
# 3- ResNet50 Model ve Hassas Eğitim
# ===============================================
print(f"\n--- ResNet50 Hazırlanıyor (Sınıf Sayısı: {len(class_names)}) ---")

model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, len(class_names))
model = model.to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=1e-5)
criterion = nn.CrossEntropyLoss()

best_acc = 0
epochs = 15 # Daha iyi yakınsama için 15 epoch idealdir

for epoch in range(epochs):
    model.train()
    t_loss, t_correct, t_total = 0, 0, 0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)

        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, lbls)
        loss.backward()
        optimizer.step()

        t_loss += loss.item()
        t_correct += (out.argmax(1) == lbls).sum().item()
        t_total += lbls.size(0)

    # Validation süreci
    model.eval()
    v_correct, v_total = 0, 0
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            v_correct += (model(imgs).argmax(1) == lbls).sum().item()
            v_total += lbls.size(0)

    val_acc = v_correct / v_total
    print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {t_loss/len(train_loader):.4f} | Train Acc: {t_correct/t_total:.4f} | Val Acc: {val_acc:.4f}")

    # En iyi modeli kaydet
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "ResNet50_best.pth")
        print(f"--> ⭐ Yeni En İyi Model Kaydedildi (Acc: {val_acc:.4f})")

print("\n✅ Eğitim Tamamlandı! ResNet50_best.pth dosyasını kullanabilirsin.")

Mounted at /content/drive

--- ResNet50 Hazırlanıyor (Sınıf Sayısı: 4) ---
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 175MB/s]


Epoch [1/15] | Train Loss: 1.0711 | Train Acc: 0.6910 | Val Acc: 0.8101
--> ⭐ Yeni En İyi Model Kaydedildi (Acc: 0.8101)
Epoch [2/15] | Train Loss: 0.4491 | Train Acc: 0.8697 | Val Acc: 0.8482
--> ⭐ Yeni En İyi Model Kaydedildi (Acc: 0.8482)
Epoch [3/15] | Train Loss: 0.2966 | Train Acc: 0.8993 | Val Acc: 0.8856
--> ⭐ Yeni En İyi Model Kaydedildi (Acc: 0.8856)
Epoch [4/15] | Train Loss: 0.2276 | Train Acc: 0.9207 | Val Acc: 0.9115
--> ⭐ Yeni En İyi Model Kaydedildi (Acc: 0.9115)
Epoch [5/15] | Train Loss: 0.1741 | Train Acc: 0.9396 | Val Acc: 0.9359
--> ⭐ Yeni En İyi Model Kaydedildi (Acc: 0.9359)
Epoch [6/15] | Train Loss: 0.1364 | Train Acc: 0.9554 | Val Acc: 0.9413
--> ⭐ Yeni En İyi Model Kaydedildi (Acc: 0.9413)
Epoch [7/15] | Train Loss: 0.1167 | Train Acc: 0.9617 | Val Acc: 0.9611
--> ⭐ Yeni En İyi Model Kaydedildi (Acc: 0.9611)
Epoch [8/15] | Train Loss: 0.0964 | Train Acc: 0.9706 | Val Acc: 0.9573
Epoch [9/15] | Train Loss: 0.0809 | Train Acc: 0.9741 | Val Acc: 0.9733
--> ⭐ Yen